# Bitcoin OTC: concepts and source audit
This notebook audits the source and defines the graph views used in the project. Each rating is a directed, signed, timestamped observation: **source rates target**. A rating is not a verified trade or a fraud label. We first check the source file, then build explicitly defined graph views.

## 1. Locate and read the original source
Keep the compressed SNAP file unchanged. The SHA-256 checksum lets teammates verify that they analysed the same bytes. The path works when Jupyter starts either in the project folder or in `notebooks/`.

In [1]:
from pathlib import Path
import hashlib
import pandas as pd
import networkx as nx

DATA_NAME = 'soc-sign-bitcoinotc.csv.gz'
PROJECT_DIR = Path.cwd() if (Path.cwd() / 'data' / DATA_NAME).exists() else Path.cwd().parent
DATA_PATH = PROJECT_DIR / 'data' / DATA_NAME
assert DATA_PATH.exists(), f'Download the SNAP file to {DATA_PATH}'
source_sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
ratings = pd.read_csv(DATA_PATH, header=None, names=['source', 'target', 'rating', 'time'])
ratings['time_utc'] = pd.to_datetime(ratings['time'], unit='s', utc=True)
print('Source SHA-256:', source_sha256)
print('Rows:', len(ratings))
display(ratings.head())

Source SHA-256: 6424ac981dad3a019f697fc1b9fcd85c19d8d9f039797758e9ffb6fea100c373
Rows: 35592


,source,target,rating,time,time_utc
0,6,2,4,1.289242e+09,2010-11-08 18:45:11.728359938+00:00
1,6,5,2,1.289242e+09,2010-11-08 18:45:41.533780098+00:00
2,1,15,1,1.289243e+09,2010-11-08 19:05:40.390490055+00:00
3,4,3,7,1.289245e+09,2010-11-08 19:41:17.369750023+00:00
4,13,16,8,1.289254e+09,2010-11-08 22:10:54.447459936+00:00


## 2. Check what is actually observed
A node is a user appearing as rater or ratee. An edge is one observed rating. Check missing values, self-ratings, repeated source–target pairs, sign balance, and the time span before constructing an undirected network of positive ratings. These checks must pass or be explained; do not silently discard records.

In [2]:
NODES = set(ratings['source']) | set(ratings['target'])
audit = {
    'rows': len(ratings),
    'nodes': len(NODES),
    'missing_cells': int(ratings.isna().sum().sum()),
    'self_ratings': int((ratings['source'] == ratings['target']).sum()),
    'repeated_directed_pairs': int(ratings.duplicated(['source', 'target']).sum()),
    'positive_ratings': int((ratings['rating'] > 0).sum()),
    'negative_ratings': int((ratings['rating'] < 0).sum()),
    'zero_ratings': int((ratings['rating'] == 0).sum()),
    'first_utc': ratings['time_utc'].min(),
    'last_utc': ratings['time_utc'].max(),
}
display(pd.Series(audit, name='source audit'))
display(ratings.groupby([ratings['time_utc'].dt.year, ratings['rating'] > 0]).size().unstack(fill_value=0))

rows                                                     35592
nodes                                                     5881
missing_cells                                                0
self_ratings                                                 0
repeated_directed_pairs                                      0
positive_ratings                                         32029
negative_ratings                                          3563
zero_ratings                                                 0
first_utc                  2010-11-08 18:45:11.728359938+00:00
last_utc                   2016-01-25 01:12:03.757280111+00:00
Name: source audit, dtype: object

rating,False,True
time_utc,,
2010,0,142
2011,155,7603
2012,810,8622
2013,1844,11138
2014,670,3555
2015,82,929
2016,2,40


## 3. Define the primary graph
The positive graph retains ratings greater than zero. Its directed **in-degree** counts distinct users who gave a positive rating to a target; it does not measure the amount of money traded or the number of transactions. We keep all observed users as nodes so filtering to positive edges does not silently erase users who only appear in negative ratings.

In [3]:
positive = ratings.loc[ratings['rating'] > 0].copy()
positive_directed_network = nx.DiGraph()
positive_directed_network.add_nodes_from(NODES)
for row in positive.itertuples(index=False):
    positive_directed_network.add_edge(
        row.source, row.target,
        rating=int(row.rating),
        weight=int(row.rating),
        time_unix=float(row.time),
    )
print('Nodes:', positive_directed_network.number_of_nodes(), 'Positive edges:', positive_directed_network.number_of_edges())
print('Mean in-degree:', sum(dict(positive_directed_network.in_degree()).values()) / positive_directed_network.number_of_nodes())
print('Weakly connected components:', nx.number_weakly_connected_components(positive_directed_network))

Nodes: 5881 Positive edges: 32029
Mean in-degree: 5.446182622003061
Weakly connected components: 317


## 4. Time as a possible focused investigation

The timestamps support the project's focused temporal analysis. A cumulative graph at date $t$ contains ratings **recorded by** that date. It is an observed history, not evidence that every old rating remained an active relationship. See the third notebook for year-end concentration and the event-level target-choice benchmark.